# A100 SU(3) combined $T_1^{+-}$ / string-tension / OP-12 pipeline

This notebook uses **one GPU-resident pure-$SU(3)$ Wilson ensemble** for two measurement branches:

1. **Glueball branch:** exact-flat-vector-projected $T_1^{+-}$ correlators at $\Gamma,X,M,R$, a multi-APE variational basis, Wilson loops, spatial torelons, blocked bootstrap, and pilot cosh fits.
2. **OP-12 branch:** defect-seeded Hodge observable
   $$\theta(U)=v_0\lambda_{\max}(\Pi_D P M^{-1}P\Pi_D)$$
   evaluated with an exact lattice Fourier symbol for $PM^{-1}P$, rather than nested conjugate-gradient solves.

## Run

1. Select **Runtime → Change runtime type → A100 GPU**.
2. Run the code cell below. No edits are required.
3. Results are written to `/content/A100_SU3_COMBINED_RUN/`.

The defaults are a validation-scale production pilot. GPU output is floating-point numerical evidence; the rational strong-coupling certificates remain separate CPU proofs.

In [1]:
#!/usr/bin/env python3
"""
A100 SU(3) COMBINED PIPELINE
============================
One Colab-ready JAX program sharing a single pure-SU(3) Wilson ensemble between:

  A. T1^{+-} glueball spectroscopy + Wilson loops + torelon/string-tension proxy
  B. OP-12 defect/Hodge theta screening using an FFT implementation of
         theta(U) = v0 * lambda_max(Pi_D P M^{-1} P Pi_D)

Key corrections relative to the two source pilots
-------------------------------------------------
* The T1^{+-} target branch is projected with the exact flat-band vector:
      O_flat(k) proportional to
      (e^{ik_x}-1) B_x + (e^{ik_y}-1) B_y + (e^{ik_z}-1) B_z.
* OP-12 uses the exact Fourier symbols of P and P M^{-1} P.  It does not run
  nested conjugate-gradient solves inside power iteration.
* A full Metropolis sweep is JIT-compiled and returns one acceptance scalar,
  avoiding 24 device-to-host synchronizations per sweep.
* Multiple APE levels are retained as a variational operator basis.
* Bootstrap resampling is blocked using the plaquette autocorrelation estimate.
* Checkpoint/resume, hard numerical gates, raw arrays, JSON summaries, and plots
  are included.

Scientific scope
----------------
This is floating-point Euclidean Monte Carlo evidence.  It is not an exact
strong-coupling certificate and is not yet the anisotropic Hamiltonian limit.

Colab
------
Choose Runtime -> Change runtime type -> A100 GPU, then run the notebook cell.
No edits are required. Environment variables can override defaults for smoke
runs, e.g.:
  COMB_L=4 COMB_LT=8 COMB_THERM=2 COMB_NMEAS=2 COMB_GAP=1 \
  COMB_THETA_EVERY=1 COMB_ALLOW_CPU=1 python a100_su3_combined_t1pm_op12.py
"""
from __future__ import annotations

import json
import math
import os
import platform
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

try:
    import jax
    import jax.numpy as jnp
    from jax import lax, random
except Exception as exc:
    raise RuntimeError(
        "JAX is required. In Colab select Runtime > Change runtime type > A100 GPU."
    ) from exc

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eigh
from scipy.optimize import curve_fit

jax.config.update("jax_enable_x64", True)


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.lower().strip() in {"1", "true", "yes", "on", "y"}


def env_int_tuple(name: str, default: str) -> Tuple[int, ...]:
    return tuple(sorted({int(x.strip()) for x in os.environ.get(name, default).split(",") if x.strip()}))


def env_float_tuple(name: str, default: str) -> Tuple[float, ...]:
    return tuple(float(x.strip()) for x in os.environ.get(name, default).split(",") if x.strip())


@dataclass(frozen=True)
class Config:
    L: int = env_int("COMB_L", 8)
    LT: int = env_int("COMB_LT", 16)
    beta: float = env_float("COMB_BETA", 5.50)
    seed: int = env_int("COMB_SEED", 20260614)

    therm_sweeps: int = env_int("COMB_THERM", 100)
    n_measurements: int = env_int("COMB_NMEAS", 80)
    sweeps_between: int = env_int("COMB_GAP", 3)

    epsilon: float = env_float("COMB_EPS", 0.24)
    target_accept: float = env_float("COMB_TARGET_ACCEPT", 0.52)
    adapt_every: int = env_int("COMB_ADAPT_EVERY", 10)
    reunit_every: int = env_int("COMB_REUNIT_EVERY", 20)

    ape_alpha: float = env_float("COMB_APE_ALPHA", 0.45)
    ape_levels: Tuple[int, ...] = env_int_tuple("COMB_APE_LEVELS", "0,3,6")
    wilson_every: int = env_int("COMB_WILSON_EVERY", 4)

    theta_every: int = env_int("COMB_THETA_EVERY", 4)
    theta_deltas: Tuple[float, ...] = env_float_tuple("COMB_DELTAS", "0.7,0.9,1.1")
    theta_m2: float = env_float("COMB_M2", 0.5)
    theta_v0: float = env_float("COMB_V0", 1.0)
    theta_power_iterations: int = env_int("COMB_POWER_ITERS", 60)

    checkpoint_every: int = env_int("COMB_CHECKPOINT_EVERY", 8)
    bootstrap_samples: int = env_int("COMB_BOOTSTRAP", 200)
    hot_start: bool = env_bool("COMB_HOT_START", True)
    allow_cpu: bool = env_bool("COMB_ALLOW_CPU", False)

    @property
    def shape4(self) -> Tuple[int, int, int, int]:
        return (self.LT, self.L, self.L, self.L)

    @property
    def volume(self) -> int:
        return self.LT * self.L**3


CFG = Config()
if CFG.L % 2:
    raise ValueError("COMB_L must be even so X, M, and R are lattice momenta.")
if not CFG.ape_levels or CFG.ape_levels[0] != 0:
    raise ValueError("COMB_APE_LEVELS must include 0 as its first level.")
if any(b < a for a, b in zip(CFG.ape_levels, CFG.ape_levels[1:])):
    raise ValueError("COMB_APE_LEVELS must be increasing.")

ROOT = Path("/content/A100_SU3_COMBINED_RUN") if Path("/content").exists() else Path.cwd() / "A100_SU3_COMBINED_RUN"
ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = ROOT / "checkpoint_latest.npz"
RAW_NPZ = ROOT / "raw_measurements.npz"
SUMMARY_JSON = ROOT / "summary.json"
LOG_TXT = ROOT / "run.log"


class Tee:
    def __init__(self, path: Path):
        self.stdout = sys.stdout
        self.file = path.open("a", encoding="utf-8")

    def write(self, text: str) -> None:
        self.stdout.write(text)
        self.file.write(text)
        self.file.flush()

    def flush(self) -> None:
        self.stdout.flush()
        self.file.flush()


sys.stdout = Tee(LOG_TXT)

CDTYPE = jnp.complex128
RDTYPE = jnp.float64
I3 = jnp.eye(3, dtype=CDTYPE)
AXES4 = (0, 1, 2, 3)
ORIENTATIONS = tuple((mu, nu) for mu in range(4) for nu in range(mu + 1, 4))
SPATIAL_ORIENTATIONS = ((2, 3), (1, 3), (1, 2))


def dagger(a: jax.Array) -> jax.Array:
    return jnp.swapaxes(jnp.conj(a), -1, -2)


def trace3(a: jax.Array) -> jax.Array:
    return jnp.trace(a, axis1=-2, axis2=-1)


def shift_field(a: jax.Array, offset: Tuple[int, int, int, int]) -> jax.Array:
    """output[x] = input[x + offset]."""
    return jnp.roll(a, shift=tuple(-int(v) for v in offset), axis=AXES4)


def unit_offset(mu: int, sign: int = 1) -> Tuple[int, int, int, int]:
    out = [0, 0, 0, 0]
    out[mu] = sign
    return tuple(out)


def add_offset(*offsets: Tuple[int, int, int, int]) -> Tuple[int, int, int, int]:
    return tuple(sum(o[i] for o in offsets) for i in range(4))


def link_at(U: jax.Array, mu: int, offset: Tuple[int, int, int, int]) -> jax.Array:
    return shift_field(U[..., mu, :, :], offset)


def project_to_su3_batch(M: jax.Array) -> jax.Array:
    u, _, vh = jnp.linalg.svd(M, full_matrices=False)
    q = u @ vh
    detq = jnp.linalg.det(q)
    return q.at[..., :, 2].multiply(jnp.conj(detq)[..., None])


def reunitarize_gram_schmidt(U: jax.Array) -> jax.Array:
    c0 = U[..., :, 0]
    c1 = U[..., :, 1]
    c0 = c0 / jnp.maximum(jnp.linalg.norm(c0, axis=-1, keepdims=True), 1e-30)
    c1 = c1 - jnp.sum(jnp.conj(c0) * c1, axis=-1, keepdims=True) * c0
    c1 = c1 / jnp.maximum(jnp.linalg.norm(c1, axis=-1, keepdims=True), 1e-30)
    c2 = jnp.conj(jnp.cross(c0, c1))
    return jnp.stack((c0, c1, c2), axis=-1)


def haar_su3(key: jax.Array, leading_shape: Tuple[int, ...]) -> jax.Array:
    kr, ki = random.split(key)
    z = random.normal(kr, leading_shape + (3, 3), dtype=RDTYPE)
    z = z + 1j * random.normal(ki, leading_shape + (3, 3), dtype=RDTYPE)
    q, r = jnp.linalg.qr(z)
    diag = jnp.diagonal(r, axis1=-2, axis2=-1)
    phase = diag / jnp.where(jnp.abs(diag) > 0, jnp.abs(diag), 1.0)
    q = q * jnp.conj(phase)[..., None, :]
    detq = jnp.linalg.det(q)
    return q.at[..., :, 2].multiply(jnp.conj(detq)[..., None]).astype(CDTYPE)


# -----------------------------------------------------------------------------
# Gauge sampler
# -----------------------------------------------------------------------------

def staple(U: jax.Array, mu: int, spatial_only: bool = False) -> jax.Array:
    result = jnp.zeros(U.shape[:4] + (3, 3), dtype=CDTYPE)
    directions = (1, 2, 3) if spatial_only else (0, 1, 2, 3)
    emu = unit_offset(mu, +1)
    for nu in directions:
        if nu == mu:
            continue
        enu = unit_offset(nu, +1)
        mnu = unit_offset(nu, -1)
        sf = link_at(U, nu, emu) @ dagger(link_at(U, mu, enu)) @ dagger(link_at(U, nu, (0, 0, 0, 0)))
        sb = dagger(link_at(U, nu, add_offset(emu, mnu))) @ dagger(link_at(U, mu, mnu)) @ link_at(U, nu, mnu)
        result = result + sf + sb
    return result


def su2_proposal(key: jax.Array, epsilon: float, subgroup: int) -> jax.Array:
    ka, kt = random.split(key)
    axis = random.normal(ka, CFG.shape4 + (3,), dtype=RDTYPE)
    axis = axis / jnp.maximum(jnp.linalg.norm(axis, axis=-1, keepdims=True), 1e-30)
    theta = random.uniform(kt, CFG.shape4, minval=-epsilon, maxval=epsilon, dtype=RDTYPE)
    s = jnp.sin(theta)
    a0 = jnp.cos(theta)
    a1, a2, a3 = axis[..., 0] * s, axis[..., 1] * s, axis[..., 2] * s
    r00, r01 = a0 + 1j * a3, a2 + 1j * a1
    r10, r11 = -a2 + 1j * a1, a0 - 1j * a3
    R = jnp.broadcast_to(I3, CFG.shape4 + (3, 3))
    i, j = ((0, 1), (0, 2), (1, 2))[subgroup]
    R = R.at[..., i, i].set(r00)
    R = R.at[..., i, j].set(r01)
    R = R.at[..., j, i].set(r10)
    R = R.at[..., j, j].set(r11)
    return R


coords = jnp.indices(CFG.shape4)
PARITY = (jnp.sum(coords, axis=0) & 1).astype(jnp.int32)


def sweep_impl(U: jax.Array, key: jax.Array, beta: float, epsilon: float) -> Tuple[jax.Array, jax.Array, jax.Array]:
    accepted = jnp.array(0.0, dtype=RDTYPE)
    for mu in range(4):
        for parity in (0, 1):
            for subgroup in range(3):
                kp, ka, key = random.split(key, 3)
                S = staple(U, mu, spatial_only=False)
                old = U[..., mu, :, :]
                proposal = su2_proposal(kp, epsilon, subgroup) @ old
                delta = (beta / 3.0) * (jnp.real(trace3(proposal @ S)) - jnp.real(trace3(old @ S)))
                logu = jnp.log(random.uniform(ka, CFG.shape4, minval=1e-300, maxval=1.0, dtype=RDTYPE))
                active = PARITY == parity
                take = active & (logu < delta)
                U = U.at[..., mu, :, :].set(jnp.where(take[..., None, None], proposal, old))
                accepted = accepted + jnp.sum(take)
    attempts = 4 * 2 * 3 * (CFG.volume // 2)
    return U, key, accepted / attempts


sweep_device = jax.jit(sweep_impl)
reunitarize_all = jax.jit(reunitarize_gram_schmidt)


def plaquette_matrix(U: jax.Array, mu: int, nu: int) -> jax.Array:
    emu, enu = unit_offset(mu, +1), unit_offset(nu, +1)
    return link_at(U, mu, (0, 0, 0, 0)) @ link_at(U, nu, emu) @ dagger(link_at(U, mu, enu)) @ dagger(link_at(U, nu, (0, 0, 0, 0)))


@jax.jit
def mean_plaquette(U: jax.Array) -> jax.Array:
    total = 0.0
    for mu, nu in ORIENTATIONS:
        total = total + jnp.mean(jnp.real(trace3(plaquette_matrix(U, mu, nu))) / 3.0)
    return total / len(ORIENTATIONS)


@jax.jit
def max_unitarity_residual(U: jax.Array) -> jax.Array:
    ident = jnp.eye(3, dtype=CDTYPE)
    return jnp.max(jnp.abs(U @ dagger(U) - ident))


# -----------------------------------------------------------------------------
# Smearing, Wilson loops, torelons, and exact flat-branch T1^{+-} projection
# -----------------------------------------------------------------------------

def ape_one_step(U: jax.Array, alpha: float) -> jax.Array:
    newU = U
    for mu in (1, 2, 3):
        st = staple(U, mu, spatial_only=True)
        candidate = (1.0 - alpha) * U[..., mu, :, :] + (alpha / 4.0) * st
        newU = newU.at[..., mu, :, :].set(project_to_su3_batch(candidate))
    return newU


ape_one_step_device = jax.jit(ape_one_step)


def smeared_levels(U: jax.Array) -> List[jax.Array]:
    out = [U]
    cur = U
    current_step = 0
    for target in CFG.ape_levels[1:]:
        while current_step < target:
            cur = ape_one_step_device(cur, CFG.ape_alpha)
            current_step += 1
        out.append(cur)
    return out


def path_link(U: jax.Array, direction: int, offset: List[int], sign: int) -> Tuple[jax.Array, List[int]]:
    if sign > 0:
        link = link_at(U, direction, tuple(offset))
        new_offset = offset.copy()
        new_offset[direction] += 1
        return link, new_offset
    new_offset = offset.copy()
    new_offset[direction] -= 1
    return dagger(link_at(U, direction, tuple(new_offset))), new_offset


def rectangular_loop_impl(U: jax.Array, mu: int, nu: int, r: int, t: int) -> jax.Array:
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3))
    offset = [0, 0, 0, 0]
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, +1); M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, +1); M = M @ lk
    for _ in range(r):
        lk, offset = path_link(U, mu, offset, -1); M = M @ lk
    for _ in range(t):
        lk, offset = path_link(U, nu, offset, -1); M = M @ lk
    return jnp.mean(jnp.real(trace3(M)) / 3.0)


rectangular_loop = jax.jit(rectangular_loop_impl, static_argnames=("mu", "nu", "r", "t"))


def polyakov_field_impl(U: jax.Array, mu: int, length: int) -> jax.Array:
    M = jnp.broadcast_to(I3, U.shape[:4] + (3, 3))
    offset = [0, 0, 0, 0]
    for _ in range(length):
        M = M @ link_at(U, mu, tuple(offset))
        offset[mu] += 1
    return trace3(M) / 3.0


polyakov_field = jax.jit(polyakov_field_impl, static_argnames=("mu", "length"))

MOMENTA = {"G": (0, 0, 0), "X": (1, 0, 0), "M": (1, 1, 0), "R": (1, 1, 1)}
MOMENTUM_NAMES = tuple(MOMENTA.keys())
xyz = np.indices((CFG.L, CFG.L, CFG.L))
phases = []
weights = []
for name in MOMENTUM_NAMES:
    bits = MOMENTA[name]
    phases.append(np.exp(-1j * np.pi * (bits[0] * xyz[0] + bits[1] * xyz[1] + bits[2] * xyz[2])))
    if name == "G":
        w = np.ones(3, dtype=np.complex128) / math.sqrt(3.0)
    else:
        w = np.array([np.exp(1j * np.pi * b) - 1.0 for b in bits], dtype=np.complex128)
        w /= np.linalg.norm(w)
    weights.append(w)
PHASES = jnp.asarray(np.stack(phases), dtype=CDTYPE)
FLAT_WEIGHTS = jnp.asarray(np.stack(weights), dtype=CDTYPE)


@jax.jit
def t1pm_components_and_flat(U_sm: jax.Array) -> Tuple[jax.Array, jax.Array]:
    """Return components[k,t,c] and exact target branch flat[k,t]."""
    p_yz = jnp.imag(trace3(plaquette_matrix(U_sm, 2, 3)))
    p_xz = jnp.imag(trace3(plaquette_matrix(U_sm, 1, 3)))
    p_xy = jnp.imag(trace3(plaquette_matrix(U_sm, 1, 2)))
    B = jnp.stack((p_yz, -p_xz, p_xy), axis=-1)
    components = jnp.einsum("kxyz,txyzc->ktc", PHASES, B, optimize=True) / math.sqrt(CFG.L**3)
    flat = jnp.einsum("kc,ktc->kt", FLAT_WEIGHTS, components, optimize=True)
    return components, flat


@jax.jit
def spatial_torelon_operators(U_sm: jax.Array) -> jax.Array:
    values = []
    for mu in (1, 2, 3):
        field = polyakov_field(U_sm, mu, CFG.L)
        values.append(jnp.mean(field, axis=(1, 2, 3)))
    return jnp.stack(values, axis=-1)


# -----------------------------------------------------------------------------
# OP-12 FFT Hodge projector
# -----------------------------------------------------------------------------

def make_fourier_symbols() -> Tuple[jax.Array, jax.Array]:
    grids = jnp.meshgrid(
        *[2.0 * jnp.pi * jnp.fft.fftfreq(n) for n in CFG.shape4], indexing="ij"
    )
    g = jnp.stack([jnp.exp(1j * k) - 1.0 for k in grids], axis=-1).astype(CDTYPE)
    g2 = jnp.sum(jnp.abs(g) ** 2, axis=-1).astype(RDTYPE)
    return g, g2


G_SYMBOL, G2_SYMBOL = make_fourier_symbols()


def hodge_project_fft(f: jax.Array) -> jax.Array:
    """Orthogonal projector onto ker d0^* for a real/complex 1-cochain."""
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    invg2 = jnp.where(G2_SYMBOL > 1e-28, 1.0 / G2_SYMBOL, 0.0)
    FP = F - G_SYMBOL * (inner * invg2)[..., None]
    return jnp.fft.ifftn(FP, axes=AXES4)


def pm_inv_fft(f: jax.Array, beta: float, m2: float) -> jax.Array:
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    invg2 = jnp.where(G2_SYMBOL > 1e-28, 1.0 / G2_SYMBOL, 0.0)
    FP = F - G_SYMBOL * (inner * invg2)[..., None]
    denom = m2 + (beta / 6.0) * G2_SYMBOL
    return jnp.fft.ifftn(FP / denom[..., None], axes=AXES4)


def apply_M_fft(f: jax.Array, beta: float, m2: float) -> jax.Array:
    F = jnp.fft.fftn(f, axes=AXES4)
    inner = jnp.sum(jnp.conj(G_SYMBOL) * F, axis=-1)
    L1F = G2_SYMBOL[..., None] * F - G_SYMBOL * inner[..., None]
    return jnp.fft.ifftn(m2 * F + (beta / 6.0) * L1F, axes=AXES4)


@jax.jit
def plaquette_defect_fields(U: jax.Array) -> jax.Array:
    vals = []
    for mu, nu in ORIENTATIONS:
        vals.append(1.0 - jnp.real(trace3(plaquette_matrix(U, mu, nu))) / 3.0)
    return jnp.stack(vals, axis=-1)  # [t,x,y,z,6]


def defect_link_mask(defects: jax.Array, delta: float) -> Tuple[jax.Array, jax.Array]:
    bad = defects > delta
    masks = [jnp.zeros(CFG.shape4, dtype=jnp.bool_) for _ in range(4)]
    for oi, (mu, nu) in enumerate(ORIENTATIONS):
        p = bad[..., oi]
        masks[mu] = masks[mu] | p | shift_field(p, unit_offset(nu, -1))
        masks[nu] = masks[nu] | p | shift_field(p, unit_offset(mu, -1))
    return jnp.stack(masks, axis=-1).astype(RDTYPE), jnp.mean(bad.astype(RDTYPE))


def theta_power_impl(mask: jax.Array, key: jax.Array, beta: float, m2: float, v0: float, n_iter: int) -> Tuple[jax.Array, jax.Array]:
    x = random.normal(key, CFG.shape4 + (4,), dtype=RDTYPE) * mask
    n0 = jnp.linalg.norm(x)
    x = jnp.where(n0 > 0, x / jnp.maximum(n0, 1e-300), x)

    def body(_, state):
        x0, _lam = state
        y = mask * jnp.real(pm_inv_fft(mask * x0, beta, m2))
        ny = jnp.linalg.norm(y)
        lam = jnp.sum(x0 * y)
        x1 = jnp.where(ny > 0, y / jnp.maximum(ny, 1e-300), x0)
        return x1, lam

    x, lam = lax.fori_loop(0, n_iter, body, (x, jnp.array(0.0, dtype=RDTYPE)))
    y = mask * jnp.real(pm_inv_fft(mask * x, beta, m2))
    lam = jnp.sum(x * y)
    return v0 * lam, jnp.mean(mask)


theta_power = jax.jit(theta_power_impl, static_argnames=("n_iter",))


def op12_measure(U: jax.Array, key: jax.Array, delta: float) -> Tuple[jax.Array, jax.Array, jax.Array, jax.Array]:
    defects = plaquette_defect_fields(U)
    mask, rho_p = defect_link_mask(defects, delta)
    theta, rho_l = theta_power(mask, key, CFG.beta, CFG.theta_m2, CFG.theta_v0, CFG.theta_power_iterations)
    nlinks = jnp.sum(mask)
    return theta, rho_p, rho_l, nlinks


# -----------------------------------------------------------------------------
# Hard gates
# -----------------------------------------------------------------------------

def run_hard_gates() -> Dict[str, float]:
    print("Running hard numerical gates...")
    key = random.PRNGKey(CFG.seed + 991)
    Uid = jnp.broadcast_to(I3, CFG.shape4 + (4, 3, 3))
    p = float(mean_plaquette(Uid))
    if abs(p - 1.0) > 2e-13:
        raise AssertionError(f"cold plaquette gate failed: {p}")
    st = staple(Uid, 0)
    st_res = float(jnp.max(jnp.abs(st - 6.0 * I3)))
    if st_res > 2e-13:
        raise AssertionError(f"cold staple gate failed: {st_res}")

    key, k1, k2 = random.split(key, 3)
    f = random.normal(k1, CFG.shape4 + (4,), dtype=RDTYPE)
    Pf = hodge_project_fft(f)
    PPf = hodge_project_fft(Pf)
    p2 = float(jnp.linalg.norm(PPf - Pf) / (1.0 + jnp.linalg.norm(Pf)))
    if p2 > 5e-11:
        raise AssertionError(f"FFT P^2 gate failed: {p2}")

    phi = random.normal(k2, CFG.shape4, dtype=RDTYPE)
    grad = jnp.stack([jnp.roll(phi, -1, axis=mu) - phi for mu in range(4)], axis=-1)
    pg = hodge_project_fft(grad)
    grad_res = float(jnp.linalg.norm(pg) / (1.0 + jnp.linalg.norm(grad)))
    if grad_res > 5e-11:
        raise AssertionError(f"FFT P grad gate failed: {grad_res}")

    y = Pf
    z = pm_inv_fft(y, CFG.beta, CFG.theta_m2)
    mz = apply_M_fft(z, CFG.beta, CFG.theta_m2)
    minv_res = float(jnp.linalg.norm(mz - y) / (1.0 + jnp.linalg.norm(y)))
    if minv_res > 2e-10:
        raise AssertionError(f"FFT PM^-1 gate failed: {minv_res}")

    result = {"cold_plaquette": p, "cold_staple_residual": st_res, "P2_residual": p2,
              "P_gradient_residual": grad_res, "PM_inverse_residual": minv_res}
    print(json.dumps(result, indent=2))
    print("Hard gates: PASS")
    return result


# -----------------------------------------------------------------------------
# Statistics
# -----------------------------------------------------------------------------

def integrated_autocorr_time(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if len(x) < 4:
        return float("nan")
    x = x - x.mean()
    var = np.dot(x, x) / len(x)
    if var <= 0:
        return 0.5
    tau = 0.5
    for lag in range(1, min(len(x) // 2, 1000)):
        rho = np.dot(x[:-lag], x[lag:]) / ((len(x) - lag) * var)
        if rho <= 0:
            break
        tau += rho
    return float(tau)


def periodic_correlator(data: np.ndarray, subtract_mean: bool = True) -> np.ndarray:
    z = np.asarray(data)
    if z.ndim == 2:
        z = z[..., None]
    if subtract_mean:
        z = z - z.mean(axis=(0, 1), keepdims=True)
    lt = z.shape[1]
    return np.array([np.real(np.mean(np.roll(z, -tau, axis=1) * np.conj(z))) for tau in range(lt)])


def blocked_indices(ncfg: int, block: int, rng: np.random.Generator) -> np.ndarray:
    block = max(1, min(block, ncfg))
    starts = np.arange(0, ncfg, block)
    blocks = [np.arange(s, min(s + block, ncfg)) for s in starts]
    chosen = rng.integers(0, len(blocks), size=len(blocks))
    idx = np.concatenate([blocks[i] for i in chosen])
    if len(idx) < ncfg:
        idx = np.resize(idx, ncfg)
    return idx[:ncfg]


def blocked_bootstrap_correlator(data: np.ndarray, block: int, nboot: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    central = periodic_correlator(data)
    if len(data) < 2 or nboot < 2:
        return central, np.full_like(central, np.nan)
    rng = np.random.default_rng(seed)
    boots = [periodic_correlator(data[blocked_indices(len(data), block, rng)]) for _ in range(nboot)]
    return central, np.std(np.asarray(boots), axis=0, ddof=1)


def correlation_matrix(data: np.ndarray) -> np.ndarray:
    """data[cfg,op,t,component] -> C[t,op,op]."""
    z = np.asarray(data)
    z = z - z.mean(axis=(0, 2), keepdims=True)
    ncfg, nop, lt, ncomp = z.shape
    C = np.empty((lt, nop, nop), dtype=np.complex128)
    for tau in range(lt):
        zs = np.roll(z, -tau, axis=2)
        C[tau] = np.einsum("natc,nbtc->ab", zs, np.conj(z), optimize=True) / (ncfg * lt * ncomp)
        C[tau] = 0.5 * (C[tau] + C[tau].conj().T)
    return C


def principal_correlator(C: np.ndarray, t0: int = 1, cut: float = 1e-8) -> np.ndarray:
    C0 = np.real_if_close(C[t0]).real
    w, v = np.linalg.eigh(0.5 * (C0 + C0.T))
    keep = w > max(cut * np.max(np.abs(w)), 1e-14)
    if not np.any(keep):
        return np.full(C.shape[0], np.nan)
    W = v[:, keep] / np.sqrt(w[keep])[None, :]
    out = []
    for Ct in C:
        M = W.T @ np.real_if_close(Ct).real @ W
        vals = np.linalg.eigvalsh(0.5 * (M + M.T))
        out.append(float(np.max(vals)))
    return np.asarray(out)


def effective_mass_cosh(c: np.ndarray) -> np.ndarray:
    out = np.full(len(c), np.nan)
    for t in range(1, len(c) - 1):
        if c[t] != 0:
            arg = (c[t - 1] + c[t + 1]) / (2.0 * c[t])
            if arg >= 1:
                out[t] = np.arccosh(arg)
    return out


def fit_periodic_cosh(c: np.ndarray, err: np.ndarray, lt: int, tmin: int = 1, tmax: int | None = None) -> Dict[str, float] | None:
    if tmax is None:
        tmax = min(lt // 2, 6)
    t = np.arange(tmin, tmax + 1)
    y = np.asarray(c[t], dtype=float)
    e = np.asarray(err[t], dtype=float)
    good = np.isfinite(y) & np.isfinite(e) & (e > 0) & (y > 0)
    if np.count_nonzero(good) < 3:
        return None
    t, y, e = t[good], y[good], e[good]

    def model(tt, amp, energy):
        return amp * (np.exp(-energy * tt) + np.exp(-energy * (lt - tt)))

    e0 = max(1e-3, float(np.log(y[0] / y[min(1, len(y) - 1)])) if len(y) > 1 and y[1] > 0 else 0.5)
    try:
        popt, pcov = curve_fit(model, t, y, sigma=e, absolute_sigma=True, p0=(max(y[0], 1e-12), e0),
                               bounds=([0.0, 1e-6], [np.inf, 20.0]), maxfev=20000)
        pred = model(t, *popt)
        chi2 = float(np.sum(((y - pred) / e) ** 2))
        dof = max(1, len(y) - 2)
        return {"amplitude": float(popt[0]), "energy": float(popt[1]),
                "energy_error": float(np.sqrt(max(pcov[1, 1], 0.0))), "chi2_dof": chi2 / dof,
                "tmin": int(tmin), "tmax": int(tmax)}
    except Exception:
        return None


def creutz_from_wilson(samples: List[Dict[str, float]]) -> Dict[str, float]:
    if not samples:
        return {}
    keys = sorted(set.intersection(*[set(s) for s in samples]))
    means = {k: float(np.mean([row[k] for row in samples])) for k in keys}
    out = {f"mean_{k}": v for k, v in means.items()}
    for prefix in ("raw", "sm"):
        for r in (1, 2):
            for t in (1, 2, 3):
                names = [f"{prefix}_W{r}_{t}", f"{prefix}_W{r+1}_{t}", f"{prefix}_W{r}_{t+1}", f"{prefix}_W{r+1}_{t+1}"]
                if all(n in means and means[n] > 0 for n in names):
                    ratio = means[names[3]] * means[names[0]] / (means[names[1]] * means[names[2]])
                    if ratio > 0:
                        out[f"{prefix}_chi_{r}_{t}"] = float(-math.log(ratio))
    return out


# -----------------------------------------------------------------------------
# Persistence and plots
# -----------------------------------------------------------------------------

def save_checkpoint(U, key, epsilon, sweep_count, plaquettes, acceptances, t1_components, t1_flat,
                    torelons, wilson_samples, theta_records):
    tmp = CHECKPOINT.with_suffix(".tmp.npz")
    np.savez_compressed(
        tmp,
        U=np.asarray(jax.device_get(U)), key=np.asarray(jax.device_get(key)), epsilon=np.array(epsilon),
        sweep_count=np.array(sweep_count), plaquettes=np.asarray(plaquettes), acceptances=np.asarray(acceptances),
        t1_components=np.asarray(t1_components), t1_flat=np.asarray(t1_flat), torelons=np.asarray(torelons),
        wilson_json=np.array(json.dumps(wilson_samples)), theta_json=np.array(json.dumps(theta_records)),
        config_json=np.array(json.dumps(asdict(CFG))),
    )
    tmp.replace(CHECKPOINT)
    print(f"checkpoint: {CHECKPOINT} ({len(plaquettes)} measurements)")


def load_checkpoint():
    if not CHECKPOINT.exists():
        return None
    d = np.load(CHECKPOINT, allow_pickle=False)
    old = json.loads(str(d["config_json"]))
    current = json.loads(json.dumps(asdict(CFG)))
    for name in ("L", "LT", "beta", "seed", "ape_levels", "theta_deltas"):
        if old.get(name) != current.get(name):
            print(f"Ignoring checkpoint: config mismatch in {name}")
            return None
    return {
        "U": jnp.asarray(d["U"], dtype=CDTYPE), "key": jnp.asarray(d["key"]), "epsilon": float(d["epsilon"]),
        "sweep_count": int(d["sweep_count"]), "plaquettes": list(np.asarray(d["plaquettes"])),
        "acceptances": list(np.asarray(d["acceptances"])), "t1_components": list(np.asarray(d["t1_components"])),
        "t1_flat": list(np.asarray(d["t1_flat"])), "torelons": list(np.asarray(d["torelons"])),
        "wilson_samples": json.loads(str(d["wilson_json"])), "theta_records": json.loads(str(d["theta_json"])),
    }


def make_plots(plaquette, acceptance, target_corr, target_err, tor_corr, tor_err, theta_records):
    plt.figure(figsize=(8, 4)); plt.plot(plaquette, lw=1); plt.xlabel("measurement"); plt.ylabel("average plaquette")
    plt.tight_layout(); plt.savefig(ROOT / "plaquette_history.png", dpi=180); plt.close()

    plt.figure(figsize=(8, 4)); plt.plot(acceptance, lw=1); plt.axhline(CFG.target_accept, ls="--", lw=1)
    plt.xlabel("sweep"); plt.ylabel("acceptance"); plt.tight_layout(); plt.savefig(ROOT / "acceptance_history.png", dpi=180); plt.close()

    plt.figure(figsize=(8, 5))
    for name in MOMENTUM_NAMES:
        c, e = target_corr[name], target_err[name]
        tt = np.arange(min(CFG.LT // 2 + 1, len(c))); den = c[0] if c[0] else 1.0
        plt.errorbar(tt, c[tt] / den, yerr=e[tt] / abs(den), marker="o", ms=3, capsize=2, label=name)
    plt.yscale("symlog", linthresh=1e-6); plt.xlabel(r"$\tau$"); plt.ylabel(r"$C_{T_1^{+-}}(\tau)/C(0)$")
    plt.legend(); plt.tight_layout(); plt.savefig(ROOT / "t1pm_exact_branch_correlators.png", dpi=180); plt.close()

    plt.figure(figsize=(8, 5))
    for name in MOMENTUM_NAMES:
        meff = effective_mass_cosh(target_corr[name]); tt = np.arange(1, min(CFG.LT // 2, len(meff) - 1))
        plt.plot(tt, meff[tt], marker="o", ms=3, label=name)
    plt.xlabel(r"$\tau$"); plt.ylabel("cosh effective energy"); plt.legend(); plt.tight_layout()
    plt.savefig(ROOT / "t1pm_exact_branch_effective_energies.png", dpi=180); plt.close()

    plt.figure(figsize=(8, 5)); tt = np.arange(min(CFG.LT // 2 + 1, len(tor_corr))); den = tor_corr[0] if tor_corr[0] else 1.0
    plt.errorbar(tt, tor_corr[tt] / den, yerr=tor_err[tt] / abs(den), marker="o", ms=3, capsize=2)
    plt.yscale("symlog", linthresh=1e-7); plt.xlabel(r"$\tau$"); plt.ylabel(r"$C_{tor}(\tau)/C(0)$")
    plt.tight_layout(); plt.savefig(ROOT / "torelon_correlator.png", dpi=180); plt.close()

    if theta_records:
        plt.figure(figsize=(8, 5))
        for delta in CFG.theta_deltas:
            rows = [r for r in theta_records if abs(r["delta"] - delta) < 1e-12]
            if rows:
                plt.plot([r["measurement"] for r in rows], [r["theta"] for r in rows], marker="o", ms=3, label=f"delta={delta}")
        plt.xlabel("measurement"); plt.ylabel(r"$\theta$"); plt.legend(); plt.tight_layout()
        plt.savefig(ROOT / "op12_theta_history.png", dpi=180); plt.close()


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def main() -> None:
    print("=" * 100)
    print("A100 SU(3) COMBINED T1^{+-} SPECTROSCOPY + STRING TENSION + OP-12 FFT SCREEN")
    print("=" * 100)
    print("Config:", json.dumps(asdict(CFG), indent=2))
    print("Python:", platform.python_version(), "JAX:", jax.__version__)
    print("Devices:", jax.devices())
    print("Output:", ROOT)
    has_gpu = any(d.platform == "gpu" for d in jax.devices())
    if not has_gpu and not CFG.allow_cpu:
        raise RuntimeError("No JAX GPU detected. Select an A100 runtime or set COMB_ALLOW_CPU=1 for a tiny smoke test.")

    gates = run_hard_gates()
    state = load_checkpoint()
    if state is None:
        key = random.PRNGKey(CFG.seed); key, ki = random.split(key)
        U = haar_su3(ki, CFG.shape4 + (4,)) if CFG.hot_start else jnp.broadcast_to(I3, CFG.shape4 + (4, 3, 3))
        epsilon, sweep_count = CFG.epsilon, 0
        plaquettes, acceptances, t1_components, t1_flat, torelons = [], [], [], [], []
        wilson_samples, theta_records = [], []
    else:
        print("Resuming from", CHECKPOINT)
        U, key, epsilon, sweep_count = state["U"], state["key"], state["epsilon"], state["sweep_count"]
        plaquettes, acceptances = state["plaquettes"], state["acceptances"]
        t1_components, t1_flat, torelons = state["t1_components"], state["t1_flat"], state["torelons"]
        wilson_samples, theta_records = state["wilson_samples"], state["theta_records"]

    print("Compiling representative kernels...")
    t0 = time.time()
    U, key, a0 = sweep_device(U, key, CFG.beta, epsilon)
    levels = smeared_levels(U)
    _ = t1pm_components_and_flat(levels[-1])
    key, kt = random.split(key)
    _ = op12_measure(U, kt, CFG.theta_deltas[0])
    jax.block_until_ready(U)
    print(f"warmup {time.time()-t0:.1f}s, plaquette={float(mean_plaquette(U)):.8f}, acceptance={float(a0):.3f}")

    if not plaquettes:
        print("\nThermalization")
        recent = []
        for s in range(CFG.therm_sweeps):
            U, key, acc_dev = sweep_device(U, key, CFG.beta, epsilon)
            acc = float(acc_dev); sweep_count += 1; acceptances.append(acc); recent.append(acc)
            if CFG.reunit_every and sweep_count % CFG.reunit_every == 0:
                U = reunitarize_all(U)
            if (s + 1) % CFG.adapt_every == 0:
                ma = float(np.mean(recent[-CFG.adapt_every:])); epsilon = float(np.clip(epsilon * math.exp(0.8 * (ma - CFG.target_accept)), 0.015, 1.2))
                print(f"therm {s+1:4d}/{CFG.therm_sweeps}: P={float(mean_plaquette(U)):.8f}, acc={ma:.3f}, eps={epsilon:.4f}")
        save_checkpoint(U, key, epsilon, sweep_count, plaquettes, acceptances, t1_components, t1_flat, torelons, wilson_samples, theta_records)

    print("\nProduction")
    start = len(plaquettes); wall = time.time()
    for m in range(start, CFG.n_measurements):
        gap_acc = []
        for _ in range(CFG.sweeps_between):
            U, key, acc_dev = sweep_device(U, key, CFG.beta, epsilon)
            acc = float(acc_dev); sweep_count += 1; acceptances.append(acc); gap_acc.append(acc)
            if CFG.reunit_every and sweep_count % CFG.reunit_every == 0:
                U = reunitarize_all(U)

        p = float(mean_plaquette(U)); levels = smeared_levels(U)
        comp_levels, flat_levels = [], []
        for Us in levels:
            comp, flat = t1pm_components_and_flat(Us)
            comp_levels.append(np.asarray(jax.device_get(comp)))
            flat_levels.append(np.asarray(jax.device_get(flat)))
        tor = np.asarray(jax.device_get(spatial_torelon_operators(levels[-1])))
        plaquettes.append(p); t1_components.append(np.stack(comp_levels)); t1_flat.append(np.stack(flat_levels)); torelons.append(tor)

        if CFG.wilson_every > 0 and m % CFG.wilson_every == 0:
            row = {}
            for prefix, Uf in (("raw", U), ("sm", levels[-1])):
                for r in (1, 2, 3):
                    for t in (1, 2, 3, 4):
                        row[f"{prefix}_W{r}_{t}"] = float(np.mean([float(rectangular_loop(Uf, mu, 0, r, t)) for mu in (1, 2, 3)]))
            wilson_samples.append(row)

        if CFG.theta_every > 0 and m % CFG.theta_every == 0:
            for delta in CFG.theta_deltas:
                key, kth = random.split(key)
                th, rp, rl, nd = op12_measure(U, kth, delta)
                theta_records.append({"measurement": m, "delta": delta, "theta": float(th), "rho_plaquette": float(rp),
                                      "rho_link": float(rl), "n_defect_links": int(nd)})

        rate = (time.time() - wall) / max(1, m + 1 - start)
        print(f"meas {m+1:4d}/{CFG.n_measurements}: P={p:.8f}, acc={np.mean(gap_acc):.3f}, eps={epsilon:.4f}, {rate:.2f}s/meas")
        if (m + 1) % CFG.checkpoint_every == 0 or m + 1 == CFG.n_measurements:
            ures = float(max_unitarity_residual(U))
            if ures > 1e-8:
                raise AssertionError(f"link unitarity gate failed: {ures}")
            save_checkpoint(U, key, epsilon, sweep_count, plaquettes, acceptances, t1_components, t1_flat, torelons, wilson_samples, theta_records)

    plaq = np.asarray(plaquettes, float); acc = np.asarray(acceptances, float)
    comps = np.asarray(t1_components, np.complex128)  # cfg,sm,k,t,c
    flats = np.asarray(t1_flat, np.complex128)        # cfg,sm,k,t
    tors = np.asarray(torelons, np.complex128)        # cfg,t,dir

    tau = integrated_autocorr_time(plaq)
    block = max(1, int(math.ceil(2.0 * tau))) if np.isfinite(tau) else 1
    target_corr, target_err, fits, gevp = {}, {}, {}, {}
    for ik, name in enumerate(MOMENTUM_NAMES):
        # Gamma: average the three T1 components. Others: exact flat-vector scalar.
        target = comps[:, -1, ik, :, :] if name == "G" else flats[:, -1, ik, :, None]
        c, e = blocked_bootstrap_correlator(target, block, CFG.bootstrap_samples, CFG.seed + 100 + ik)
        target_corr[name], target_err[name] = c, e
        fits[name] = fit_periodic_cosh(c, e, CFG.LT)

        # Variational basis across APE levels.
        basis = comps[:, :, ik, :, :] if name == "G" else flats[:, :, ik, :, None]
        C = correlation_matrix(basis)
        gevp[name] = principal_correlator(C).tolist()

    tor_corr, tor_err = blocked_bootstrap_correlator(tors, block, CFG.bootstrap_samples, CFG.seed + 200)
    tor_fit = fit_periodic_cosh(tor_corr, tor_err, CFG.LT)
    sigma_proxy = None
    if tor_fit is not None:
        E = tor_fit["energy"]
        sigma_proxy = (E + math.pi / (3.0 * CFG.L)) / CFG.L

    creutz = creutz_from_wilson(wilson_samples)
    np.savez_compressed(
        RAW_NPZ, config_json=np.array(json.dumps(asdict(CFG))), plaquette=plaq, acceptance=acc,
        t1_components=comps, t1_flat=flats, torelons=tors,
        t1_corr=np.stack([target_corr[n] for n in MOMENTUM_NAMES]),
        t1_corr_err=np.stack([target_err[n] for n in MOMENTUM_NAMES]),
        torelon_corr=tor_corr, torelon_corr_err=tor_err,
        momentum_names=np.asarray(MOMENTUM_NAMES), flat_weights=np.asarray(FLAT_WEIGHTS),
        wilson_json=np.array(json.dumps(wilson_samples)), theta_json=np.array(json.dumps(theta_records)),
    )

    summary = {
        "title": "Combined A100 SU(3) T1+-/string-tension/OP-12 FFT pilot", "status": "COMPLETE",
        "scope": "isotropic Euclidean floating-point Monte Carlo; not an exact or Hamiltonian-limit certificate",
        "config": asdict(CFG), "devices": [str(d) for d in jax.devices()], "hard_gates": gates,
        "measurements": len(plaq), "mean_plaquette": float(np.mean(plaq)), "plaquette_tau_int": tau,
        "bootstrap_block_measurements": block, "mean_acceptance": float(np.mean(acc)), "final_epsilon": epsilon,
        "t1_exact_branch_cosh_fits": fits, "t1_variational_principal_correlators": gevp,
        "torelon_cosh_fit": tor_fit, "sigma_a2_single_L_proxy": sigma_proxy,
        "creutz_and_wilson": creutz, "op12_theta_records": theta_records,
        "files": {"raw": str(RAW_NPZ), "checkpoint": str(CHECKPOINT), "log": str(LOG_TXT)},
    }
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    make_plots(plaq, acc, target_corr, target_err, tor_corr, tor_err, theta_records)

    print("\n" + "=" * 100); print("RUN COMPLETE"); print("=" * 100)
    print(f"mean plaquette              = {summary['mean_plaquette']:.10f}")
    print(f"plaquette tau_int           = {tau:.3f} measurements")
    print(f"blocked-bootstrap block     = {block}")
    print(f"mean acceptance             = {summary['mean_acceptance']:.4f}")
    print("T1+- exact-branch fits:")
    for name in MOMENTUM_NAMES:
        print(f"  {name}: {fits[name]}")
    print("torelon fit:", tor_fit)
    print("single-L sigma a^2 proxy:", sigma_proxy)
    if theta_records:
        for delta in CFG.theta_deltas:
            vals = [r["theta"] for r in theta_records if abs(r["delta"] - delta) < 1e-12]
            if vals:
                print(f"OP12 delta={delta}: median theta={np.median(vals):.8f}, max={np.max(vals):.8f}")
    print("\nOutputs:")
    for path in sorted(ROOT.iterdir()):
        print(" ", path)
    print("\nInterpretation: all GPU outputs are numerical evidence. Exact rational theorem certificates remain CPU-verified.")


if __name__ == "__main__":
    main()


A100 SU(3) COMBINED T1^{+-} SPECTROSCOPY + STRING TENSION + OP-12 FFT SCREEN
Config: {
  "L": 8,
  "LT": 16,
  "beta": 5.5,
  "seed": 20260614,
  "therm_sweeps": 100,
  "n_measurements": 80,
  "sweeps_between": 3,
  "epsilon": 0.24,
  "target_accept": 0.52,
  "adapt_every": 10,
  "reunit_every": 20,
  "ape_alpha": 0.45,
  "ape_levels": [
    0,
    3,
    6
  ],
  "wilson_every": 4,
  "theta_every": 4,
  "theta_deltas": [
    0.7,
    0.9,
    1.1
  ],
  "theta_m2": 0.5,
  "theta_v0": 1.0,
  "theta_power_iterations": 60,
  "checkpoint_every": 8,
  "bootstrap_samples": 200,
  "hot_start": true,
  "allow_cpu": false
}
Python: 3.12.13 JAX: 0.7.2
Devices: [CudaDevice(id=0)]
Output: /content/A100_SU3_COMBINED_RUN
Running hard numerical gates...
{
  "cold_plaquette": 1.0,
  "cold_staple_residual": 0.0,
  "P2_residual": 4.1957558259825525e-16,
  "P_gradient_residual": 2.2807460882685333e-16,
  "PM_inverse_residual": 7.397775667409443e-16
}
Hard gates: PASS
Compiling representative kernels...
